## Summary

This EDA notebook provides insights into the customer sentiment dataset:

- **Dataset**: 1,429 processed reviews with bilingual content (Hindi/Hinglish)
- **Language Mix**: Combination of Devanagari script, Hinglish, and English
- **Aspects**: Multiple aspect categories (food, service, ambiance, price, etc.)
- **Sentiments**: Positive, negative, and neutral classifications
- **Splits**: 80% train, 10% validation, 10% test

### Key Observations:
- The dataset contains restaurant reviews in multiple languages
- Aspects are extracted and labeled from review text
- Sentiment labels are available for aspect-based sentiment analysis
- Data is preprocessed and ready for model training with PyABSA

### Next Steps:
1. Run `notebooks/02_model_training.ipynb` for model development
2. Train ABSA models using the PyABSA framework
3. Evaluate model performance on test set
4. Deploy model API using FastAPI

In [ ]:
# Sentiment Distribution Analysis
sentiments = []
try:
    with open(train_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    for line in lines:
        if '$$$' in line:
            parts = line.strip().split('$$$')
            if len(parts) >= 3:
                sentiment = parts[2].strip()
                sentiments.append(sentiment)
    
    sentiment_counts = Counter(sentiments)
    sorted_sentiments = sorted(sentiment_counts.items(), key=lambda x: x[1], reverse=True)
    
    print("="*60)
    print("SENTIMENT DISTRIBUTION")
    print("="*60)
    print(f"\nTotal sentiment annotations: {len(sentiments)}")
    print(f"Unique sentiment classes: {len(sentiment_counts)}\n")
    
    # Display statistics
    print("Sentiment Class Distribution:")
    print("-" * 40)
    for sentiment, count in sorted_sentiments:
        pct = count / len(sentiments) * 100
        print(f"  {sentiment:15s}: {count:5d} ({pct:5.1f}%)")
    
    # Check for imbalance
    max_count = max(count for _, count in sorted_sentiments)
    min_count = min(count for _, count in sorted_sentiments)
    imbalance_ratio = max_count / min_count if min_count > 0 else 0
    
    print(f"\nClass Imbalance Ratio: {imbalance_ratio:.2f}x")
    if imbalance_ratio > 3:
        print("⚠ Note: Significant class imbalance detected!")
        print("  Consider using weighted loss or resampling techniques during training")
    else:
        print("✓ Relatively balanced classes")
    
    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    sentiments_names = [s[0] for s in sorted_sentiments]
    sentiments_counts = [s[1] for s in sorted_sentiments]
    colors = sns.color_palette("Set2", len(sentiments_names))
    
    axes[0].bar(sentiments_names, sentiments_counts, color=colors)
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Sentiment Class Distribution (Counts)')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (name, count) in enumerate(zip(sentiments_names, sentiments_counts)):
        axes[0].text(i, count + 5, str(count), ha='center', va='bottom')
    
    # Pie chart
    axes[1].pie(sentiments_counts, labels=sentiments_names, autopct='%1.1f%%',
               colors=colors, startangle=90)
    axes[1].set_title('Sentiment Distribution (%)')
    
    plt.tight_layout()
    plt.show()
    
    # Train/Val/Test distribution comparison
    print("\n" + "="*60)
    print("SPLIT DISTRIBUTION")
    print("="*60)
    
    val_path = Path("../data/processed/val_data.txt")
    test_path = Path("../data/processed/test_data.txt")
    
    splits_info = {
        'Train (80%)': train_path,
        'Val (10%)': val_path,
        'Test (10%)': test_path
    }
    
    for split_name, split_file in splits_info.items():
        if split_file.exists():
            try:
                with open(split_file, 'r', encoding='utf-8') as f:
                    split_lines = f.readlines()
                print(f"\n{split_name}: {len(split_lines)} records")
            except Exception as e:
                print(f"\n{split_name}: Error reading file - {e}")
        else:
            print(f"\n{split_name}: File not found - {split_file}")
    
except Exception as e:
    print(f"Error in sentiment analysis: {e}")
    print("Make sure to run the feature engineering pipeline first")

## 6. Sentiment Balance Analysis

Analyzing sentiment distribution and identifying potential class imbalance issues

In [ ]:
# Load feature metadata to analyze aspect distribution
metadata_path = Path("../data/processed/features_metadata.json")
train_path = Path("../data/processed/train_data.txt")

if train_path.exists():
    print("="*60)
    print("ASPECT DISTRIBUTION ANALYSIS")
    print("="*60)
    
    # Extract aspects from PyABSA format: text$$$[aspect]$$$sentiment
    aspects = []
    try:
        with open(train_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        for line in lines:
            if '$$$' in line:
                parts = line.strip().split('$$$')
                if len(parts) >= 2:
                    aspect = parts[1].strip('[]')
                    aspects.append(aspect)
        
        aspect_counts = Counter(aspects)
        
        # Sort by frequency
        sorted_aspects = sorted(aspect_counts.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nTotal unique aspects: {len(aspect_counts)}")
        print(f"Total aspect mentions: {len(aspects)}")
        print("\nTop 10 Aspects:")
        print("-" * 40)
        for aspect, count in sorted_aspects[:10]:
            pct = count / len(aspects) * 100
            print(f"  {aspect:20s}: {count:4d} ({pct:5.1f}%)")
        
        # Visualization
        if len(sorted_aspects) > 0:
            fig, axes = plt.subplots(1, 2, figsize=(15, 5))
            
            # Bar chart
            top_n = 12
            top_aspects = sorted_aspects[:top_n]
            aspects_names = [a[0] for a in top_aspects]
            aspects_counts = [a[1] for a in top_aspects]
            
            axes[0].barh(aspects_names, aspects_counts, color='steelblue')
            axes[0].set_xlabel('Frequency')
            axes[0].set_title('Top 12 Aspect Distribution')
            axes[0].invert_yaxis()
            
            # Pie chart
            other_count = sum(count for _, count in sorted_aspects[top_n:])
            pie_labels = aspects_names + ['Others']
            pie_counts = aspects_counts + [other_count]
            colors = sns.color_palette("husl", len(pie_labels))
            axes[1].pie(pie_counts, labels=pie_labels, autopct='%1.1f%%', 
                       colors=colors, startangle=90)
            axes[1].set_title('Aspect Distribution (% of total)')
            
            plt.tight_layout()
            plt.show()
            
    except Exception as e:
        print(f"Error reading training data: {e}")
        print("Make sure to run the feature engineering pipeline first")
else:
    print(f"✗ Training data file not found: {train_path}")
    print("Make sure to run: python src/features/build_features.py")

## 5. Aspect Distribution Analysis

In [ ]:
def detect_script(text):
    """Detect if text contains Devanagari (Hindi) characters"""
    devanagari_count = len([c for c in text if '\u0900' <= c <= '\u097f'])
    return "Hindi/Devanagari" if devanagari_count > len(text) * 0.3 else "Roman/Hinglish/English"

# Display random samples
if 'text' in df.columns:
    print("="*60)
    print("SAMPLE REVIEWS - Language Mix")
    print("="*60)
    
    sample_indices = np.random.choice(len(df), size=min(10, len(df)), replace=False)
    
    for idx, i in enumerate(sample_indices, 1):
        text = df.iloc[i]['text']
        script_type = detect_script(text)
        # Truncate long text
        display_text = text if len(text) <= 100 else text[:100] + "..."
        print(f"\n{idx}. [{script_type}]")
        print(f"   {display_text}")
    
    print("\n" + "="*60)
    print(f"Dataset Language Distribution:")
    scripts = [detect_script(text) for text in df['text']]
    script_counts = Counter(scripts)
    for script, count in script_counts.items():
        pct = count / len(df) * 100
        print(f"  {script}: {count} ({pct:.1f}%)")
else:
    print("✗ 'text' column not found in dataset")

## 4. Hindi/Hinglish Examples

Displaying sample reviews to show the language mix in the dataset (Hindi, Hinglish, English)

In [ ]:
# Dataset Information
print("="*60)
print("DATASET STATISTICS")
print("="*60)
print(f"\nTotal Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}")

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✓ No missing values")
else:
    print(missing[missing > 0])

print("\n--- Text Length Statistics ---")
if 'text' in df.columns:
    text_lengths = df['text'].str.len()
    print(f"Average text length: {text_lengths.mean():.1f} characters")
    print(f"Min text length: {text_lengths.min()} characters")
    print(f"Max text length: {text_lengths.max()} characters")
    print(f"Median text length: {text_lengths.median():.1f} characters")

print("\n--- First 3 Rows ---")
display(df.head(3))

## 3. Dataset Statistics

In [ ]:
# Load processed data
data_path = Path("../data/processed/processed_data.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"✓ Loaded processed data from {data_path}")
else:
    print(f"✗ File not found: {data_path}")
    print("Loading sample data instead...")
    df = pd.DataFrame()

print(f"\nDataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
display(df.head())

## 2. Load Processed Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")

## 1. Import Required Libraries

# Customer Sentiment Analysis - Exploratory Data Analysis
## Hindi/Hinglish Restaurant Reviews Dataset

This notebook explores the processed customer sentiment dataset, examining:
- Dataset structure and statistics
- Language mix (Hindi, Hinglish, English)
- Aspect distribution across reviews
- Sentiment balance and class imbalance issues